# Train the models

Trains every (variant, seed) pair below for one dataset. With the same seed, both variants see identical blocks, points and rotations, and these are also the blocks RandLA-Net saw with that seed. The only difference between the variants is the `u_tilde` input column.

## Setup

In [ ]:
import os, sys
ROOT = os.path.abspath('..')
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 50)

from src import paths
print('project folder:', paths.PROJECT_ROOT)

import json
import torch
from src.config import loadConfig
from src.train import trainRun

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu')

In [ ]:
#DATASET   = 'toronto3d'
VARIANTS  = ['baseline', 'uncertainty']
SEEDS     = [42, 43, 44]
#OVERRIDES = []
#OVERRIDES = ['training.batchBlocks=8', 'training.blocksPerEpoch=2000']

## Train

In [ ]:
PLAN = {
    'toronto3d'  : ['training.batchBlocks=8'],
    'paris_lille': ['training.batchBlocks=8'],
    'ahn3'       : ['training.batchBlocks=8'],
}

done = []
for seed in SEEDS:
    for ds, ov in PLAN.items():
        cfg = loadConfig(ds, ov)
        for variant in VARIANTS:
            print(f'\n===== {ds} / {variant} / seed {seed} =====')
            trainRun(cfg, variant, seed, device)
            done.append((ds, variant, seed))
            torch.cuda.empty_cache()
print('\nfinished:', done)


'''
Above one is a one shott execution, below one is a single one. Choose accordingly.
Pick the above one for an overnight run.
'''

'''
cfg = loadConfig(DATASET, OVERRIDES)
print('runs go to:', paths.datasetOutDir(cfg))
print(json.dumps({k: cfg[k] for k in ('blocks', 'model', 'training', 'labels')}, indent=2))

done = []
for seed in SEEDS:
    for variant in VARIANTS:
        print(f'\n===== {DATASET} / {variant} / seed {seed} =====')
        trainRun(cfg, variant, seed, device)
        done.append((variant, seed))
        torch.cuda.empty_cache()
print('\nfinished:', done)
'''

## Training curves

In [ ]:
'''
#This only works per dataset
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
for variant, seed in done:
    with open(os.path.join(paths.runDir(cfg, variant, seed), 'history.json')) as f:
        h = json.load(f)
    ls = '-' if variant == 'baseline' else '--'
    axes[0].plot(h['epoch'], h['loss'], ls, label=f'{variant} s{seed}')
    axes[1].plot(h['epoch'], h['oa'],   ls, label=f'{variant} s{seed}')
axes[0].set_title(f'Training loss — {DATASET}'); axes[0].set_xlabel('epoch')
axes[1].set_title(f'Training OA — {DATASET}');   axes[1].set_xlabel('epoch')
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()
'''

dsList = list(PLAN)
fig, axes = plt.subplots(len(dsList), 2, figsize=(14, 4.5 * len(dsList)), squeeze=False)
for row, ds in enumerate(dsList):
    dsCfg = loadConfig(ds)
    for variant in VARIANTS:
        for seed in SEEDS:
            hPath = os.path.join(paths.runDir(dsCfg, variant, seed), 'history.json')
            if not os.path.exists(hPath):
                continue
            with open(hPath) as f:
                h = json.load(f)
            ls = '-' if variant == 'baseline' else '--'
            axes[row, 0].plot(h['epoch'], h['loss'], ls, label=f'{variant} s{seed}')
            axes[row, 1].plot(h['epoch'], h['oa'],   ls, label=f'{variant} s{seed}')
    axes[row, 0].set_title(f'Training loss — {ds}'); axes[row, 0].set_xlabel('epoch')
    axes[row, 1].set_title(f'Training OA — {ds}');   axes[row, 1].set_xlabel('epoch')
    for ax in axes[row]:
        ax.legend(fontsize=8)
plt.tight_layout()
plt.show()